In [1]:
import HeST as hest
import HeST.Amherst_split_cpd as examp
import numpy as np
import matplotlib.pyplot as plt
import HeST.Detection as detection
%matplotlib qt

import astropy.stats as astat
from scipy.interpolate import interp1d

In [2]:

detector = examp.Amherst_split_cpd 


QP_conditions= detector.get_surface_conditions()
nCPDs = detector.get_nCPDs()
for i in range(nCPDs):
    QP_conditions.append( (detector.get_CPD(i)).get_surface_condition() )

QP_conditions.append( detector.liquid_surface )
detector.set_QP_reflection_prob(0.)

In [4]:
pos = [0., 0., 1.5]
useMap = False
evap = hest.GetEvaporationSignal( detector, 10,*pos, useMap=useMap, debug=False, plot_3d=False)
plt.figure()
plt.hist(evap.arrivalTimes_us[0], bins=200, range=[0, 3000], histtype='step', color='r', lw=2, label = 'CPD1')
plt.hist(evap.arrivalTimes_us[1], bins=200, range=[0, 3000], histtype='step', color='b', lw=2, label = 'CPD2')
plt.legend()
plt.title('Arrival Times')
plt.xlabel('time [us]')
plt.ylabel('Counts/bin')


(10,)
(0, 1, 0, 0, 0, 0, 0, 0, 0, 0)


IndexError: too many indices for array: array is 1-dimensional, but 10 were indexed

In [ ]:
probs = np.linspace(0, .9, 10)
counts = np.empty_like(probs)
for ii, p in enumerate(probs):   
    pos = [0., 0., 1.5]
    detector.set_QP_reflection_prob(p)
    
    useMap = False
    evap = hest.GetEvaporationSignal( detector, 50000,*pos, useMap=useMap)
    counts[ii] = len(evap.arrivalTimes_us[0])

plt.plot(probs, counts)
plt.title('Logical Check. As the number of reflections goes up, so the should the number of events we have')
plt.legend()



In [ ]:
import HeST.Detection as detection
import os
import numpy as np
#The detector geometry is defined from the point of view of particle paths.
# We essentially want to define various "surface conditions" where the particle paths are obstructed
# These functions also carry a "boundary_type", so that we can keep track if the particle is obstructed by
# a CPD, or a wall, and how it may reflect off of a given wall.

def sensor1_conditions(x, y, z):
    boundary_type = "CPD0"
    radius = 3.8
    height = 3.3
    return (x*x + y*y < radius*radius) & (x>0)& (z < height)| (x*x + y*y >= radius*radius) , boundary_type

def sensor2_conditions(x, y, z):
    boundary_type = "CPD1"
    radius = 3.8
    height = 3.3
    return (x*x + y*y < radius*radius) &  (x<0)&(z < height)| (x*x + y*y >= radius*radius) , boundary_type





baseline_noise = [0., 0.]
phonon_conversion = 0.25
cpd1 = detection.VCPD(sensor1_conditions, baseline_noise, phonon_conversion)
cpd2 = detection.VCPD(sensor2_conditions, baseline_noise, phonon_conversion)





def wall_conditions(x, y, z):
    boundary_type = "XY"
    radius = 3. #cm
    height = 2.75 #cm
    return ((x*x + y*y < radius*radius) & (z < height) ) | (z > height), boundary_type

def bottom_conditions(x, y, z):
    boundary_type = "Z"
    bottom = 0. #cm
    return (z > bottom), boundary_type

def liquid_surface(x, y, z):
    boundary_type = "Liquid"
    height = 2.75 #cm
    return (z < height), boundary_type

def liquid_conditions(x, y, z):
    height = 2.75 #cm
    radius = 3. #cm
    bottom = 0. #cm
    return ((x*x + y*y < radius*radius) & (z < height) & (z > bottom))
   

Amherst_split_cpd = detection.VDetector([wall_conditions, bottom_conditions], liquid_surface=liquid_surface, liquid_conditions=liquid_conditions, CPDs=[cpd1, cpd2], adsorption_gain=6.0e-3, evaporation_eff=0.60)

 

In [ ]:
def sensor2_conditions(x, y, z):
    boundary_type = "CPD1"
    radius = 3.8
    height = 3.3
    return (x*x + y*y < radius*radius) &  (x<0)&(z < height)| (x*x + y*y >= radius*radius) , boundary_type

detector= detection.VDetector([wall_conditions, bottom_conditions], liquid_surface=liquid_surface, liquid_conditions=liquid_conditions, CPDs=[cpd1, cpd2], adsorption_gain=6.0e-3, evaporation_eff=0.60)


cpd_2_x_bounds = np.linspace(0,-3.1, 10)
signals_cpd1 = np.empty_like(cpd_2_x_bounds)
signals_cpd2 = np.empty_like(cpd_2_x_bounds)


for ii, bound in enumerate(cpd_2_x_bounds):
    pos = [0., 0., 1.5]
    def sensor2_conditions(x, y, z):
        boundary_type = "CPD1"
        radius = 3.8
        height = 3.3
        print(bound)
        return (x*x + y*y < radius*radius) &  (x<bound)&(z < height)| (x*x + y*y >= radius*radius) , boundary_type
    cpd1 = detection.VCPD(sensor1_conditions, baseline_noise, phonon_conversion)
    cpd2 = detection.VCPD(sensor2_conditions, baseline_noise, phonon_conversion)


    detector = detection.VDetector([wall_conditions, bottom_conditions], liquid_surface=liquid_surface, liquid_conditions=liquid_conditions, CPDs=[cpd1, cpd2], adsorption_gain=6.0e-3, evaporation_eff=0.60)

    useMap = False
    evap = hest.GetEvaporationSignal( detector, 40000, *pos, useMap=useMap)
    print( evap.area_eV, evap.chArea_eV, evap.coincidence, len(evap.arrivalTimes_us))
    signals_cpd1[ii] = len(evap.arrivalTimes_us[0])
    signals_cpd2[ii] = len(evap.arrivalTimes_us[1])

plt.plot(cpd_2_x_bounds, signals_cpd1, label = 'Without changing size')
plt.plot(cpd_2_x_bounds, signals_cpd2, label = 'CPD2, with changing size')
plt.legend()
plt.xlabel('Where cpd1 starts')
plt.ylabel('Number of hits per')
plt.title('Testing if cpd 1 and 2 separation is working')


In [ ]:
def GetInterpFunc():
    """Creates an linear interpolation function from data found at the file path below,, giving us the ability to convert from resistance to temperature. 
    returns:
        Interpoltion function: If input exceeds range of the data function returns a NaN"""
    data = np.loadtxt('./dispersion_data.csv', delimiter=',')
    X = data[:,0]
    Y = data[:,1]
    return interp1d(X,Y, kind = 'linear')

In [ ]:
interp = GetInterpFunc()

In [ ]:
plt.subplot(211)
data = np.loadtxt('./dispersion_data.csv', delimiter=',')
v_data = np.loadtxt('./velocity_data.csv', delimiter=',')
X = data[:,0]
Y = data[:,1]
plt.plot(X, Y, label = 'Dispersion Curve')
plt.legend()
plt.show()
plt.subplot(212)
x = v_data[:,0]
y = v_data[:,1]
plt.plot(x[:-3], y[:-3], label = 'velocity')
plt.legend()
plt.show()

In [ ]:
def k_squared_distribution(u):
    N = (4.7**3 - .15**3)/3
    c = .15**3/(3 * N)
    return (3 * N* (u + c))**(1/3)

u = np.random.uniform(size=10000)
dist = k_squared_distribution(u)
plt.hist(dist, bins = 200)
plt.title('Distribution of K-values')
plt.xlabel('momenta')
plt.ylabel('counts/bin')